# DX 704 Week 6 Project - TwizzleFlu and the Markov Decision Process

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a **Markov decision process**.
The model transition probabilities are provided in the file "`twizzleflu-transitions.tsv`" and the expected rewards are in "`twizzleflu-rewards.tsv`".
The goal for Twizzleflu is to <font color = 'cyan'>minimize the expected discomfort of the patient</font>, which is expressed as **negative rewards** in the file.

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Helper functions

### compute_qT_once

In [2]:
# one step state-action values from a value estimate.
# will use this a lot!

def compute_qT_once(R, P, gamma, v):
    """
    Compute the state-action value function Q(s, a) for one step.

        Parameters:
            - R (np.ndarray): Reward matrix of shape (num_actions, num_states).
            - P (np.ndarray): Transition probability matrix of shape (num_actions, num_states, num_states).
            - gamma (float): Discount factor (0 <= gamma <= 1).
            - v (np.ndarray): Current value function of shape (num_states,).

        Returns:
            - np.ndarray: Q-values of shape (num_actions, num_states).
    """
    return R + gamma * P @ v

### iterate_values_once

In [3]:
def iterate_values_once(R, P, gamma, v):
    """
       Perform one iteration of value update using the Bellman optimality equation.

        Parameters:
            - R (np.ndarray): Reward matrix of shape (num_actions, num_states).
            - P (np.ndarray): Transition probability matrix of shape (num_actions, num_states, num_states).
            - gamma (float): Discount factor.
            - v (np.ndarray): Current value function.

        Returns:
            - np.ndarray: Updated value function after one iteration.
    """
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

### value_iteration

In [4]:
def value_iteration(R, P, gamma = 1.0, max_iterations = 1000, tolerance = 1e-3):
    """
    Performs value iteration to compute the value function for both fixed
    and optimal policies.

    Parameters:
        - R (np.ndarray): Reward array. Can be 1D (num_states,) for a fixed policy
                          or 2D (num_actions, num_states) for an optimal policy.
        - P (np.ndarray): Transition probability matrix/tensor. Can be 2D
                          (num_states, num_states) for a fixed policy or 3D
                          (num_actions, num_states, num_states) for an optimal policy.
        - gamma (float): Discount factor (0 <= gamma <= 1). How much the agent cares about future rewards.
        - max_iterations (int): Maximum number of iterations to run.
        - tolerance (float): Convergence threshold.

    Returns:
        - np.ndarray: Value function representing expected cumulative reward for each state.
    """
    num_states  = R.shape[-1]
    v           = np.zeros(num_states)
    
    for _ in range(max_iterations):
        if R.ndim == 1:
            # Fixed policy (single action)
            v_new = compute_qT_once(R, P, gamma, v)
        elif R.ndim == 2:
            # Optimal policy (multiple actions)
            q_values    = compute_qT_once(R, P, gamma, v)
            v_new       = np.max(q_values, axis = 0)
        else:
            raise ValueError("R must be a 1D or 2D NumPy array.")
            
        if np.max(np.abs(v_new - v)) < tolerance:
            break
        v = v_new
        
    return v

### iterative_policy_evaluation

In [5]:
# copied from Implementing Iterative Policy Evaluation Video
def iterative_policy_evaluation(R, P, gamma, pi, 
                                max_iterations = 1000, tolerance = 0.001, warmstart = None
                                ):
    """
      Evaluate a fixed policy using iterative policy evaluation.

        Parameters:
        - R (np.ndarray): Reward matrix of shape (num_actions, num_states).
        - P (np.ndarray): Transition probability matrix of shape (num_actions, num_states, num_states).
        - gamma (float): Discount factor.
        - pi (np.ndarray): Policy array of shape (num_states,) with action indices.
        - max_iterations (int): Maximum number of iterations.
        - tolerance (float): Convergence threshold.
        - warmstart (np.ndarray or None): Optional initial value function.

        Returns:
        - np.ndarray: Value function for the given policy.
    """
        # factor out action choices using policy.

    # deterministic version
    n       = R.shape[-1]
    R_pi    = R[pi, np.arange(n)]
    P_pi    = P[pi, np.arange(n),:]

    # reshape to one dummy action to reuse previous example code
    R_pi = R_pi.reshape(1, *R_pi.shape)
    P_pi = P_pi.reshape(1, *P_pi.shape)

    # initial approximation v_0
    v_old = warmstart if warmstart is not None else np.zeros(R.shape[-1])

    for i in range(max_iterations):
        # compute v_{i+1}
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)

        # check if values did not change much
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    # return v_{max_iterations}
    return v_old

### policy_iteration_baselin

In [6]:
def policy_iteration_baseline(R, P, gamma, max_iterations = 1000, tolerance = 0.001):
    """
      Perform policy iteration to find the optimal policy and its value function.

        Parameters:
        - R (np.ndarray): Reward matrix of shape (num_actions, num_states).
        - P (np.ndarray): Transition probability matrix of shape (num_actions, num_states, num_states).
        - gamma (float): Discount factor.
        - max_iterations (int): Maximum number of iterations.
        - tolerance (float): Convergence threshold.

        Returns:
        - tuple: (optimal_policy, value_function)
    """
    pi_old  = np.zeros(R.shape[-1], dtype="int64")
    v_old   = iterative_policy_evaluation(R, P, gamma, pi_old)

    for i in range(max_iterations):
        # compute new policy
        pi_new = np.argmax(compute_qT_once(R, P, gamma, v_old), axis=0)
        if np.array_equal(pi_new, pi_old):
            return pi_new, v_old

        # compute new values
        v_new = iterative_policy_evaluation(R, P, gamma, pi_new)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return pi_new, v_new

        pi_old  = pi_new
        v_old   = v_new

    return pi_old, v_old

### policy_iteration_warmstart

In [7]:
def policy_iteration_warmstart(R, P, gamma, max_iterations = 1000, tolerance = 0.001):
    """
      Perform policy iteration with warm-started value function to accelerate convergence.

        Parameters:
        - R (np.ndarray): Reward matrix of shape (num_actions, num_states).
        - P (np.ndarray): Transition probability matrix of shape (num_actions, num_states, num_states).
        - gamma (float): Discount factor.
        - max_iterations (int): Maximum number of iterations.
        - tolerance (float): Convergence threshold.

        Returns:
        - tuple: (optimal_policy, value_function)
    """
    pi_old  = np.zeros(R.shape[-1], dtype="int64")
    v_old   = iterative_policy_evaluation(R, P, gamma, pi_old)

    for i in range(max_iterations):
        # compute new policy
        pi_new = np.argmax(compute_qT_once(R, P, gamma, v_old), axis = 0)
        if np.array_equal(pi_new, pi_old):
            return pi_new, v_old

        # compute new values
        v_new = iterative_policy_evaluation(R, P, gamma, pi_new, warmstart = v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return pi_new, v_new

        pi_old  = pi_new
        v_old   = v_new

    return pi_old, v_old

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Load and set up

In [8]:
transition_probabilities_df     = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
expected_rewards_df             = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

In [9]:
transition_probabilities_df

,action,state,next_state,probability
0,do-nothing,exposed-1,exposed-2,0.80
1,do-nothing,exposed-1,recovered,0.20
2,do-nothing,exposed-2,exposed-3,0.80
3,do-nothing,exposed-2,recovered,0.20
4,do-nothing,exposed-3,symptoms-1,0.80
5,do-nothing,exposed-3,recovered,0.20
6,do-nothing,symptoms-1,symptoms-1,0.70
7,do-nothing,symptoms-1,symptoms-2,0.30
8,do-nothing,symptoms-2,symptoms-2,0.70
9,do-nothing,symptoms-2,symptoms-3,0.30


In [10]:
expected_rewards_df

,action,state,reward
0,do-nothing,exposed-1,0.000
1,do-nothing,exposed-2,0.000
2,do-nothing,exposed-3,0.000
3,do-nothing,symptoms-1,-0.500
4,do-nothing,symptoms-2,-1.000
5,do-nothing,symptoms-3,-0.500
6,do-nothing,recovered,0.000
7,drink-oj,exposed-1,0.000
8,drink-oj,exposed-2,0.000
9,drink-oj,exposed-3,0.000


In [11]:
actions         = expected_rewards_df['action'].unique().tolist()
num_actions     = len(actions)
actions_idx_map = {action: idx for idx, action in enumerate(actions)}

print(f'actions: {actions}\n')
print(f'actions_idx_map: {actions_idx_map}')


actions: ['do-nothing', 'drink-oj', 'sleep-8']

actions_idx_map: {'do-nothing': 0, 'drink-oj': 1, 'sleep-8': 2}


## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
<font color = 'cyan'> Calculate the expected discomfort (not rewards) of a policy that always does nothing.</font>

Hint: for this value calculation and later ones, use *value iteration*. The analytical solution has difficulties in practice when there is no discount factor.

In [12]:
# isolating the "do-nothing" action rewards
criteria_donothing_rewards  = expected_rewards_df['action'] == 'do-nothing'
donothing_rewards_df        = expected_rewards_df[criteria_donothing_rewards].sort_values('state')
print(f'donothing_rewards_df\n{donothing_rewards_df}\n')

# Reward vector R = reward for each state under the "do-nothing" action
R_donothing     = donothing_rewards_df['reward'].to_numpy()
# R

states_idx_map  = {state: idx for idx, state in enumerate(donothing_rewards_df['state'])}
print(f'states_idx_map\n{states_idx_map}')

donothing_rewards_df
       action       state  reward
0  do-nothing   exposed-1     0.0
1  do-nothing   exposed-2     0.0
2  do-nothing   exposed-3     0.0
6  do-nothing   recovered     0.0
3  do-nothing  symptoms-1    -0.5
4  do-nothing  symptoms-2    -1.0
5  do-nothing  symptoms-3    -0.5

states_idx_map
{'exposed-1': 0, 'exposed-2': 1, 'exposed-3': 2, 'recovered': 3, 'symptoms-1': 4, 'symptoms-2': 5, 'symptoms-3': 6}


In [13]:
# isolating the "do-nothing" action transitions and their probabilities
criteria_donothing_transitions  = transition_probabilities_df['action'] == 'do-nothing'
donothing_transitions_df        = transition_probabilities_df[criteria_donothing_transitions].sort_values(['state', 'next_state'])
print(f'donothing_transitions_df\n{donothing_transitions_df}\n')


num_states            = len(states_idx_map)
P_donothing           = np.zeros((num_states, num_states)) #  7 x 7  matrix of zeros to initialize P
print(f'P initial\n{P_donothing}\n')


# Transition matrix P = transition probabilities for each state and next state under the "do-nothing" action
# the rows of P are 'state', the columns are 'next_state'... kinda like a correlation matrix

for _, row in donothing_transitions_df.iterrows():
    state       = row['state']
    next_state  = row['next_state']
    probability = row['probability']
    row_idx     = states_idx_map[state]
    col_idx     = states_idx_map[next_state]
    print(f'state: {state}, next_state: {next_state}, probability: {probability}, row_idx: {row_idx}, col_idx: {col_idx}')

    P_donothing[row_idx, col_idx] = probability

print(f'\nP final\n{P_donothing}')

donothing_transitions_df
        action       state  next_state  probability
0   do-nothing   exposed-1   exposed-2          0.8
1   do-nothing   exposed-1   recovered          0.2
2   do-nothing   exposed-2   exposed-3          0.8
3   do-nothing   exposed-2   recovered          0.2
5   do-nothing   exposed-3   recovered          0.2
4   do-nothing   exposed-3  symptoms-1          0.8
12  do-nothing   recovered   recovered          1.0
6   do-nothing  symptoms-1  symptoms-1          0.7
7   do-nothing  symptoms-1  symptoms-2          0.3
8   do-nothing  symptoms-2  symptoms-2          0.7
9   do-nothing  symptoms-2  symptoms-3          0.3
11  do-nothing  symptoms-3   recovered          0.3
10  do-nothing  symptoms-3  symptoms-3          0.7

P initial
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]

state: exposed-1, next_state: exposed-2, probability: 0.8, row_idx

In [35]:
# expected cumulative REWARD for each state under the "do-nothing" action
# gamma                   = 0.99999999
gamma                   = 1.0
reward_donothing        = value_iteration(R_donothing, P_donothing, gamma = gamma)

print(f'reward_donothing\n{reward_donothing}\n')

discomfort_donothing    = -reward_donothing
print(f'discomfort_donothing\n{discomfort_donothing}\n')

discomfort_donothing_df = pd.DataFrame({
    'state':               list(states_idx_map.keys()),
    'expected_discomfort': discomfort_donothing
})
print(f'discomfort_donothing_df\n{discomfort_donothing_df}\n')

reward_donothing
[-3.41014998 -4.26372226 -5.33061402  0.         -6.66415879 -4.99969234
 -1.66664826]

discomfort_donothing
[ 3.41014998  4.26372226  5.33061402 -0.          6.66415879  4.99969234
  1.66664826]

discomfort_donothing_df
        state  expected_discomfort
0   exposed-1             3.410150
1   exposed-2             4.263722
2   exposed-3             5.330614
3   recovered            -0.000000
4  symptoms-1             6.664159
5  symptoms-2             4.999692
6  symptoms-3             1.666648



Save the expected discomfort by state to a file "**do-nothing-discomfort.tsv**" with columns `state` and `expected_discomfort`.

In [15]:
# # save to CSV

discomfort_donothing_df.to_csv('do-nothing-discomfort.tsv', sep='\t', index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [36]:

print(f'num_states: {num_states}, num_actions: {num_actions}\n')
states        = list(states_idx_map.keys())
print(f'states (in alphabetical order): {states}\n')

## Initializing R and P matrices for all actions    
R_optimal     = np.zeros((num_actions, num_states))
P_optimal     = np.zeros((num_actions, num_states, num_states)) # apparently this is a tensor now


num_states: 7, num_actions: 3

states (in alphabetical order): ['exposed-1', 'exposed-2', 'exposed-3', 'recovered', 'symptoms-1', 'symptoms-2', 'symptoms-3']



In [37]:
## Populate the optimal Reward matrix R_optimal
for index, row in expected_rewards_df.iterrows():
    action      = row['action']
    state       = row['state']
    reward      = row['reward']
    action_idx  = actions_idx_map[action]
    state_idx   = states_idx_map[state]
    R_optimal[action_idx, state_idx] = reward

print("--- Multi-Action Reward Matrix R (Shape: 3 actions x 7 states) ---")
print("Rows are Actions, Columns are States (in alphabetical order).")
print(f'R_optimal shape: {R_optimal.shape}\n')

R_optimal_df = pd.DataFrame(R_optimal, index=actions, columns=states)
print("R_optimal_df")
R_optimal_df

--- Multi-Action Reward Matrix R (Shape: 3 actions x 7 states) ---
Rows are Actions, Columns are States (in alphabetical order).
R_optimal shape: (3, 7)

R_optimal_df


,exposed-1,exposed-2,exposed-3,recovered,symptoms-1,symptoms-2,symptoms-3
do-nothing,0.0,0.0,0.0,0.0,-0.500,-1.00,-0.500
drink-oj,0.0,0.0,0.0,0.0,-0.375,-0.75,-0.375
sleep-8,0.0,0.0,0.0,0.0,-1.000,-2.00,-1.000


In [38]:
# --- Populate the P_optimal Tensor ---

# Iterate through every row in the raw transitions data
for index, row in transition_probabilities_df.iterrows():
    action          = row['action']
    current_state   = row['state']
    next_state      = row['next_state']
    prob            = row['probability']

    # Get  numerical indices for the action (layer), current state (row), and next state (column)
    action_idx      = actions_idx_map[action]
    state_idx       = states_idx_map[current_state]
    state_next_idx  = states_idx_map[next_state]

    # Place the probability into correct P[action, current_state, next_state] slot
    P_optimal[action_idx, state_idx, state_next_idx] = prob


print("--- Multi-Action Transition Probability Tensor P_optimal (Shape: 3 actions x 7 states x 7 states) ---")
# Display each 2D transition matrix layer by layer
for i, action in enumerate(actions):
    print(f"\nTransition Matrix for ACTION: '{action}' (P[{i}, :, :])")
    print(P_optimal[i, :, :])


--- Multi-Action Transition Probability Tensor P_optimal (Shape: 3 actions x 7 states x 7 states) ---

Transition Matrix for ACTION: 'do-nothing' (P[0, :, :])
[[0.  0.8 0.  0.2 0.  0.  0. ]
 [0.  0.  0.8 0.2 0.  0.  0. ]
 [0.  0.  0.  0.2 0.8 0.  0. ]
 [0.  0.  0.  1.  0.  0.  0. ]
 [0.  0.  0.  0.  0.7 0.3 0. ]
 [0.  0.  0.  0.  0.  0.7 0.3]
 [0.  0.  0.  0.3 0.  0.  0.7]]

Transition Matrix for ACTION: 'drink-oj' (P[1, :, :])
[[0.   0.8  0.   0.2  0.   0.   0.  ]
 [0.   0.   0.8  0.2  0.   0.   0.  ]
 [0.   0.   0.   0.2  0.8  0.   0.  ]
 [0.   0.   0.   1.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.75 0.25 0.  ]
 [0.   0.   0.   0.   0.   0.75 0.25]
 [0.   0.   0.   0.25 0.   0.   0.75]]

Transition Matrix for ACTION: 'sleep-8' (P[2, :, :])
[[0.  0.5 0.  0.5 0.  0.  0. ]
 [0.  0.  0.5 0.5 0.  0.  0. ]
 [0.  0.  0.  0.5 0.5 0.  0. ]
 [0.  0.  0.  1.  0.  0.  0. ]
 [0.  0.  0.  0.  0.8 0.2 0. ]
 [0.  0.  0.  0.  0.  0.8 0.2]
 [0.  0.  0.  0.2 0.  0.  0.8]]


In [39]:
# Compute the Optimal Value Function (V*) using Value Iteration
# optimal_value_function is a vector of shape (num_states,) representing the maximum expected reward  (minimum expected discomfort) achievable from each state.
gamma       = 0.9999999999

optimal_value_function  = value_iteration(R_optimal, P_optimal, gamma = gamma)


In [40]:
#  optimal actions for each state
optimal_q_values        = compute_qT_once(R_optimal, P_optimal, gamma, optimal_value_function)
print(f'optimal_q_values.shape: {optimal_q_values.shape}\n')

optimal_actions_idx     = np.argmax(optimal_q_values, axis = 0)
print(f'optimal_actions_idx: {optimal_actions_idx}\n')

optimal_actions         = [actions[idx] for idx in optimal_actions_idx]
print(f'optimal_actions.shape: {np.array(optimal_actions).shape}\n')

minimum_discomfort_actions_df = pd.DataFrame({
    'state': states,
    'action': optimal_actions
})
minimum_discomfort_actions_df

optimal_q_values.shape: (3, 7)

optimal_actions_idx: [2 2 2 0 1 1 1]

optimal_actions.shape: (7,)



,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,drink-oj
5,symptoms-2,drink-oj
6,symptoms-3,drink-oj


Save the optimal actions for each state to a file "**minimum-discomfort-actions.tsv**" with columns `state` and `action`.

In [41]:
minimum_discomfort_actions_df.to_csv('minimum-discomfort-actions.tsv', sep='\t', index=False)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the *expected discomfort* for each state.

In [42]:
# # YOUR CHANGES HERE

# Compute the expected discomfort for each state using previous optimal policy,
results_optimal_treatment_df = pd.DataFrame({
    'state':               states,
    'action':              optimal_actions,
    'expected_discomfort': -optimal_value_function
})
results_optimal_treatment_df

,state,action,expected_discomfort
0,exposed-1,sleep-8,0.748937
1,exposed-2,sleep-8,1.498330
2,exposed-3,sleep-8,2.997378
3,recovered,do-nothing,-0.000000
4,symptoms-1,drink-oj,5.995888
5,symptoms-2,drink-oj,4.499452
6,symptoms-3,drink-oj,1.499964


Save your results in a file "minimum-discomfort-values.tsv" with columns `state` and `expected_discomfort`.

In [43]:
# # YOUR CHANGES HERE

results_optimal_treatment_df.to_csv('minimum-discomfort-values.tsv', 
                                    sep     = '\t', 
                                    columns = ['state', 'expected_discomfort'], 
                                    index   = False
                                    )

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, <font color = 'cyan'>change the reward function to always be `-1` if the current state corresponds to being sick and `0` if the current state corresponds to being better.</font>
To be clear, **the action does not matter for this reward function**.


<font color = 'plum'> In other words, transition to a new optimization objective: minimize DURATION rather than discomfort, which means that the reward function should only depend on `state`, not `action`. 

In [44]:

duration_rewards = []

for action in actions:
    for state in states:
        if state != 'recovered':
            reward = -1
        else:
            reward = 0
        duration_rewards.append(
            {'action': action,
             'state': state,
             'reward': reward
             }
        )
        
duration_rewards_df = pd.DataFrame(duration_rewards)
duration_rewards_df

,action,state,reward
0,do-nothing,exposed-1,-1
1,do-nothing,exposed-2,-1
2,do-nothing,exposed-3,-1
3,do-nothing,recovered,0
4,do-nothing,symptoms-1,-1
5,do-nothing,symptoms-2,-1
6,do-nothing,symptoms-3,-1
7,drink-oj,exposed-1,-1
8,drink-oj,exposed-2,-1
9,drink-oj,exposed-3,-1


Save your new reward function in a file "`duration-rewards.tsv`" in the same format as "twizzleflu-rewards.tsv". The columns should be `action`, `state` and `reward`.

In [45]:
# # YOUR CHANGES HERE

duration_rewards_df.to_csv('duration-rewards.tsv',
                           sep      = '\t',
                           index    = False
                           )


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu. Save the optimal actions for each state to a file "**minimum-duration-actions.tsv**" with columns `state` and `action`.

In [46]:
# YOUR CHANGES HERE
# Compute an optimal policy to minimize the DURATION of Twizzleflu. 

R_5 = duration_rewards_df.pivot(index = 'action', columns = 'state', values = 'reward')
print(f'R_5 shape: {R_5.shape}\n')
R_5 = R_5.to_numpy()


R_5 shape: (3, 7)



In [49]:
optimal_actions_idx_duration, optimal_value_function_duration = policy_iteration_warmstart(
    R       = R_5,
    P       = P_optimal,
    gamma   = 1.0
)

print(f'optimal_actions_idx_duration: {optimal_actions_idx_duration}\n')
print(f'optimal_value_function_duration: {optimal_value_function_duration}\n')


# Map optimal actions to names
optimal_actions_duration = [actions[idx] for idx in optimal_actions_idx_duration]

minimum_duration_actions_df = pd.DataFrame({
    'state': states,
    'action': optimal_actions_duration
})
minimum_duration_actions_df

optimal_actions_idx_duration: [2 2 2 0 0 0 0]

optimal_value_function_duration: [-2.99983474 -3.99975708 -5.99964334  0.         -9.99947692 -6.66660993
 -3.3333303 ]



,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,recovered,do-nothing
4,symptoms-1,do-nothing
5,symptoms-2,do-nothing
6,symptoms-3,do-nothing


Submit "**minimum-duration-actions.tsv**" in Gradescope.

In [50]:
minimum_duration_actions_df.to_csv("minimum-duration-actions.tsv", sep = "\t", index = False)

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [51]:
# YOUR CHANGES HERE

#  expected sick days (negative of value function)
expected_sick_days          = -1 * optimal_value_function_duration
print(f'expected_sick_days: {expected_sick_days}\n')

minimum_duration_days_df    = pd.DataFrame({
    'state': states,
    'expected_sick_days': expected_sick_days
})
minimum_duration_days_df

expected_sick_days: [ 2.99983474  3.99975708  5.99964334 -0.          9.99947692  6.66660993
  3.3333303 ]



,state,expected_sick_days
0,exposed-1,2.999835
1,exposed-2,3.999757
2,exposed-3,5.999643
3,recovered,-0.000000
4,symptoms-1,9.999477
5,symptoms-2,6.666610
6,symptoms-3,3.333330


Save the expected sick days for each state to a file "**minimum-duration-days.tsv**" with columns `state` and `expected_sick_days`.

In [53]:
minimum_duration_days_df.to_csv("minimum-duration-days.tsv", sep="\t", index=False)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [57]:
# YOUR CHANGES HERE
gamma = 1.0
print(f'optimal_actions_duration: {optimal_actions_duration}\n')

# Map actions to indices
pi_duration = np.array([actions_idx_map[a] for a in optimal_actions_duration])
print(f'pi_duration: {pi_duration}\n')

# Get expected discomfort using duration-minimizing policy and then negate b/c iterative_policy_evaluation computes the expected cumulative reward for each state under a fixed policy. 
expected_discomfort_duration_policy = -1 * iterative_policy_evaluation(R_optimal, P_optimal, gamma = gamma, pi = pi_duration)
print(f'expected_discomfort_duration_policy: {expected_discomfort_duration_policy}\n')


optimal_actions_duration: ['sleep-8', 'sleep-8', 'sleep-8', 'do-nothing', 'do-nothing', 'do-nothing', 'do-nothing']

pi_duration: [2 2 2 0 0 0 0]

expected_discomfort_duration_policy: [ 0.83255615  1.66551651  3.33163376 -0.          6.66415879  4.99969234
  1.66664826]



In [60]:
discomfort_comparison_df = pd.DataFrame({
    'state':               states,
    'speed_discomfort':    expected_discomfort_duration_policy,
    'minimize_discomfort': results_optimal_treatment_df['expected_discomfort']
})
discomfort_comparison_df

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.832556,0.748937
1,exposed-2,1.665517,1.498330
2,exposed-3,3.331634,2.997378
3,recovered,-0.000000,-0.000000
4,symptoms-1,6.664159,5.995888
5,symptoms-2,4.999692,4.499452
6,symptoms-3,1.666648,1.499964


Save the results to a file "**policy-comparison.tsv**" with columns `state`, `speed_discomfort`, and `minimize_discomfort`.

In [61]:

discomfort_comparison_df.to_csv("policy-comparison.tsv", sep="\t", index=False)

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.